In [2]:
# %%
# =============================================================================
# 09_rag_topk_variants.ipynb
# Financial AI Governance — RAG Top-k Ablation Study (Option A)
# Kernel : Python (llm_env)
# Input  : data/processed/dataset_final.json
#          vectordb/ (Chroma persistent stores from 02_rag_pipeline.ipynb)
# Output : results/responses/responses_rag_k1.json
#          results/responses/responses_rag_k5.json
#          results/tables/table_topk_summary.csv
# Note   : k=3 results reused from results/responses/responses_rag.json
#          Only k=1 and k=5 are newly generated here.
# =============================================================================

# %%
# =============================================================================
# Cell 1. Libraries and Environment Setup
# =============================================================================
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Directory paths — identical to 03_llm_inference.ipynb
DATA_DIR     = '../data/processed'
VDB_DIR      = '../vectordb'
RESPONSE_DIR = '../results/responses'
TABLE_DIR    = '../results/tables'

for d in [RESPONSE_DIR, TABLE_DIR]:
    os.makedirs(d, exist_ok=True)

# API setup
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
LLM_MODEL      = os.getenv('LLM_MODEL', 'gpt-4o-mini')
EMBED_MODEL    = 'text-embedding-3-small'

if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env")

client     = OpenAI(api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL, api_key=OPENAI_API_KEY)

# Top-k variants — k=3 already exists, only k=1 and k=5 are newly run
TOP_K_VARIANTS = [1, 5]

print(f"[INFO] LLM model      : {LLM_MODEL}")
print(f"[INFO] Embedding model: {EMBED_MODEL}")
print(f"[INFO] Top-k variants : {TOP_K_VARIANTS}  (k=3 reused from responses_rag.json)")


# %%
# =============================================================================
# Cell 2. Load Dataset and Vector Stores
# =============================================================================
with open(os.path.join(DATA_DIR, 'dataset_final.json'), 'r', encoding='utf-8') as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
print(f"[INFO] Dataset loaded: {len(df)} records")

# Load Chroma vector stores — identical to 03_llm_inference.ipynb
VDB_CONFIG = {
    'NIST_AI_RMF'    : {'persist_dir': os.path.join(VDB_DIR, 'nist'),          'collection': 'nist_ai_rmf'},
    'KR_AI_BASIC_ACT': {'persist_dir': os.path.join(VDB_DIR, 'kr_aibasicact'), 'collection': 'kr_aibasicact'},
    'EU_AI_ACT'      : {'persist_dir': os.path.join(VDB_DIR, 'eu_aiact'),      'collection': 'eu_aiact'},
}

vector_stores = {}
for reg_key, cfg in VDB_CONFIG.items():
    vector_stores[reg_key] = Chroma(
        collection_name    = cfg['collection'],
        embedding_function = embeddings,
        persist_directory  = cfg['persist_dir'],
    )
    count = vector_stores[reg_key]._collection.count()
    print(f"  [LOAD] {reg_key:20s} | {count} chunks")

print("[INFO] All vector stores loaded.")


# %%
# =============================================================================
# Cell 3. Prompt Templates
# =============================================================================
# Identical to 03_llm_inference.ipynb — do not modify

SYSTEM_PROMPT = """You are an expert AI governance advisor specializing in financial institution AI compliance.
Your role is to support an AI Review Committee at a financial institution by providing accurate,
regulation-grounded answers to governance questions.

When answering:
1. Cite specific regulatory provisions (article numbers, section codes) where applicable.
2. Identify the governance axis: G1 (Accuracy), G2 (Safety), G3 (Transparency), or G4 (Compliance).
3. Flag high-risk scenarios and recommend human oversight where appropriate.
4. If uncertain, state limitations clearly rather than fabricating information.
5. Keep answers concise, structured, and actionable for a compliance committee."""


def build_rag_prompt(question: str, context: str) -> str:
    return f"""Answer the following AI governance question using the regulatory context provided below.

--- REGULATORY CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}

Provide a structured answer grounded in the regulatory context above.
Cite specific article numbers or section codes from the context where applicable."""


# %%
# =============================================================================
# Cell 4. Retrieval and Inference Functions
# =============================================================================
def retrieve_context(question: str, regulation: str, k: int) -> str:
    """
    Retrieve top-k relevant chunks from the regulation-specific Chroma store.
    Identical logic to 03_llm_inference.ipynb, with explicit k parameter.
    """
    if regulation not in vector_stores:
        raise ValueError(f"[ERROR] Unknown regulation: {regulation}")
    docs    = vector_stores[regulation].similarity_search(question, k=k)
    context = "\n\n---\n\n".join([d.page_content for d in docs])
    return context


def call_llm(system_prompt: str, user_prompt: str,
             model: str = LLM_MODEL,
             temperature: float = 0.0,
             max_tokens: int = 1000) -> dict:
    """
    Call OpenAI Chat Completion API.
    Identical to 03_llm_inference.ipynb call_llm().
    """
    try:
        res = client.chat.completions.create(
            model       = model,
            temperature = temperature,
            max_tokens  = max_tokens,
            messages    = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_prompt},
            ]
        )
        return {
            'response'         : res.choices[0].message.content.strip(),
            'prompt_tokens'    : res.usage.prompt_tokens,
            'completion_tokens': res.usage.completion_tokens,
            'total_tokens'     : res.usage.total_tokens,
        }
    except Exception as e:
        return {
            'response'         : f'[ERROR] {str(e)}',
            'prompt_tokens'    : 0,
            'completion_tokens': 0,
            'total_tokens'     : 0,
        }


# %%
# =============================================================================
# Cell 5. Retrieval Quality Spot-Check
# =============================================================================
TEST_QUERIES = [
    {'question'  : 'What are the requirements for human oversight of '
                   'high-risk AI credit scoring systems?',
     'regulation': 'EU_AI_ACT'},
    {'question'  : 'What obligations apply to AI business operators '
                   'providing High-Impact AI for credit screening?',
     'regulation': 'KR_AI_BASIC_ACT'},
    {'question'  : 'How does the GOVERN function address legal and '
                   'regulatory requirements for AI systems?',
     'regulation': 'NIST_AI_RMF'},
]

print("[INFO] Retrieval Quality Spot-Check\n")
print("=" * 70)

for t in TEST_QUERIES:
    print(f"Question   : {t['question'][:70]}...")
    print(f"Regulation : {t['regulation']}")
    for k in [1, 3, 5]:
        ctx     = retrieve_context(t['question'], t['regulation'], k=k)
        chunks  = ctx.split("\n\n---\n\n")
        preview = chunks[0][:80].replace('\n', ' ')
        print(f"  k={k}: {len(chunks)} chunk(s) | {len(ctx):,} chars | "
              f"first: {preview}...")
    print("-" * 70)


# %%
# =============================================================================
# Cell 6. Run RAG Inference — k=1 and k=5
# =============================================================================
for k in TOP_K_VARIANTS:
    print(f"\n{'='*60}")
    print(f"[RUN] RAG top-k={k} inference")
    print(f"      Model: {LLM_MODEL} | Temperature: 0.0 | "
          f"Max tokens: 1000 | Top-k: {k}")
    print(f"      Total records: {len(df)}\n")

    results      = []
    total_tokens = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'RAG k={k}'):
        context     = retrieve_context(row['question'], row['regulation'], k=k)
        user_prompt = build_rag_prompt(row['question'], context)
        result      = call_llm(SYSTEM_PROMPT, user_prompt)

        # Field structure identical to 03_llm_inference.ipynb
        results.append({
            'id'               : row['id'],
            'scenario_id'      : row['scenario_id'],
            'regulation'       : row['regulation'],
            'function'         : row['function'],
            'difficulty'       : row['difficulty'],
            'financial_domain' : row['financial_domain'],
            'risk_level'       : row['risk_level'],
            'governance_axis'  : row['governance_axis'],
            'question'         : row['question'],
            'ground_truth'     : row['ground_truth'],
            'legal_basis'      : row['legal_basis'],
            'condition'        : f'rag_k{k}',
            'top_k'            : k,
            'context_used'     : context,
            'response'         : result['response'],
            'prompt_tokens'    : result['prompt_tokens'],
            'completion_tokens': result['completion_tokens'],
            'total_tokens'     : result['total_tokens'],
            'model'            : LLM_MODEL,
            'temperature'      : 0.0,
        })

        total_tokens += result['total_tokens']
        time.sleep(0.3)

        idx = len(results)
        if idx % 50 == 0:
            errors  = sum(1 for r in results if r['response'].startswith('[ERROR]'))
            avg_len = sum(len(r['response']) for r in results) / idx
            print(f"  [Checkpoint {idx:3d}/300] errors: {errors} | "
                  f"avg response: {avg_len:.0f} chars | "
                  f"tokens so far: {total_tokens:,}")

    out_path = os.path.join(RESPONSE_DIR, f'responses_rag_k{k}.json')
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    errors = sum(1 for r in results if r['response'].startswith('[ERROR]'))
    print(f"\n[SAVE] responses_rag_k{k}.json")
    print(f"[INFO] k={k} | records: {len(results)} | errors: {errors} | "
          f"total tokens: {total_tokens:,} | "
          f"estimated cost: ${total_tokens * 0.00000015:.4f}")


# %%
# =============================================================================
# Cell 7. Summary Statistics — All Top-k Conditions
# =============================================================================
ALL_CONDITIONS = {
    'rag_k1': os.path.join(RESPONSE_DIR, 'responses_rag_k1.json'),
    'rag_k3': os.path.join(RESPONSE_DIR, 'responses_rag.json'),
    'rag_k5': os.path.join(RESPONSE_DIR, 'responses_rag_k5.json'),
}

all_data = {}
for label, path in ALL_CONDITIONS.items():
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            all_data[label] = json.load(f)
        print(f"[LOAD] {label:10s}: {len(all_data[label])} records")
    else:
        print(f"[WARN] {label:10s}: file not found — {os.path.basename(path)}")

rows = []
for label, data in all_data.items():
    if not data:
        continue
    avg_ctx   = sum(len(r.get('context_used', '')) for r in data) / len(data)
    avg_resp  = sum(len(r.get('response', ''))      for r in data) / len(data)
    avg_tok   = sum(r.get('total_tokens', 0)        for r in data) / len(data)
    total_tok = sum(r.get('total_tokens', 0)        for r in data)
    errors    = sum(1 for r in data if r.get('response', '').startswith('[ERROR]'))
    rows.append({
        'Condition'           : label,
        'N'                   : len(data),
        'Top-k'               : data[0].get('top_k', 3),
        'Avg Context (chars)' : round(avg_ctx),
        'Avg Response (chars)': round(avg_resp),
        'Avg Total Tokens'    : round(avg_tok),
        'Total Tokens'        : total_tok,
        'Errors'              : errors,
        'Est. Cost ($)'       : round(total_tok * 0.00000015, 4),
    })

df_summary = pd.DataFrame(rows).sort_values('Top-k').reset_index(drop=True)
print("\n[Table] Top-k Variant Summary Statistics")
print(df_summary.to_string(index=False))

out_tbl = os.path.join(TABLE_DIR, 'table_topk_summary.csv')
df_summary.to_csv(out_tbl, index=False, encoding='utf-8-sig')
print(f"\n[SAVE] table_topk_summary.csv")

print(f"\n✅ Notebook 09 complete — Next: 10_rag_query_rewriting.ipynb")
print(f"   Then run 04_evaluation_g1_g4.ipynb on k1/k5 outputs")

[INFO] LLM model      : gpt-4o-mini
[INFO] Embedding model: text-embedding-3-small
[INFO] Top-k variants : [1, 5]  (k=3 reused from responses_rag.json)
[INFO] Dataset loaded: 300 records
  [LOAD] NIST_AI_RMF          | 11 chunks
  [LOAD] KR_AI_BASIC_ACT      | 17 chunks
  [LOAD] EU_AI_ACT            | 23 chunks
[INFO] All vector stores loaded.
[INFO] Retrieval Quality Spot-Check

Question   : What are the requirements for human oversight of high-risk AI credit s...
Regulation : EU_AI_ACT
  k=1: 1 chunk(s) | 3,338 chars | first: ### Article 14 — Human Oversight   **Chapter III, Section 2: Requirements for Hi...
  k=3: 3 chunk(s) | 11,023 chars | first: ### Article 14 — Human Oversight   **Chapter III, Section 2: Requirements for Hi...
  k=5: 5 chunk(s) | 15,129 chars | first: ### Article 14 — Human Oversight   **Chapter III, Section 2: Requirements for Hi...
----------------------------------------------------------------------
Question   : What obligations apply to AI business operator

RAG k=1:  17%|████████████                                                            | 50/300 [07:59<39:21,  9.45s/it]

  [Checkpoint  50/300] errors: 0 | avg response: 2469 chars | tokens so far: 51,156


RAG k=1:  33%|███████████████████████▋                                               | 100/300 [16:24<38:51, 11.66s/it]

  [Checkpoint 100/300] errors: 0 | avg response: 2496 chars | tokens so far: 102,991


...

  [Checkpoint 200/300] errors: 0 | avg response: 2810 chars | tokens so far: 592,356


RAG k=5:  83%|███████████████████████████████████████████████████████████▏           | 250/300 [57:14<08:47, 10.54s/it]

  [Checkpoint 250/300] errors: 0 | avg response: 2737 chars | tokens so far: 749,931


RAG k=5: 100%|█████████████████████████████████████████████████████████████████████| 300/300 [1:09:34<00:00, 13.92s/it]

  [Checkpoint 300/300] errors: 0 | avg response: 2732 chars | tokens so far: 908,394

[SAVE] responses_rag_k5.json
[INFO] k=5 | records: 300 | errors: 0 | total tokens: 908,394 | estimated cost: $0.1363
[LOAD] rag_k1    : 300 records
[LOAD] rag_k3    : 300 records
[LOAD] rag_k5    : 300 records

[Table] Top-k Variant Summary Statistics
Condition   N  Top-k  Avg Context (chars)  Avg Response (chars)  Avg Total Tokens  Total Tokens  Errors  Est. Cost ($)
   rag_k1 300      1                 1930                  2384              1066        319664       0         0.0479
   rag_k3 300      3                 6457                  2692              2044        613318       0         0.0920
   rag_k5 300      5                11304                  2732              3028        908394       0         0.1363

[SAVE] table_topk_summary.csv

✅ Notebook 09 complete — Next: 10_rag_query_rewriting.ipynb
   Then run 04_evaluation_g1_g4.ipynb on k1/k5 outputs
